In [1]:
import pandas as pd
import numpy as np
import json


In [2]:
with open('mapping_dict.json','r') as f:
    mapping = json.load(f)

time_unit_mapping      = mapping.get('time_unit_mapping', {})
temp_unit_mapping      = mapping.get('temp_unit_mapping', {})
zn_source              = mapping.get('zn_source', {})
solvent                = mapping.get('solvent', {})
zn_source_mass_mapping = mapping.get('zn_source_mass_mapping', {})
additive_name          = mapping.get('additive_name', {})
amount_unit_mapping    = mapping.get('amount_unit_mapping', {})
solvent_volume_unit    = mapping.get('solvent_volume_unit', {})
counterion_mapping     = mapping.get('counterion',{})
stirring               = mapping.get('stirring',{})
HMIM_MASS              = mapping.get('HMIM_MASS', np.nan)

ml_to_mmol = {'methanol': 0.8 / 32.04 * 1000,
 'water': 1 / 18 * 1000,
 'N,N-dimethylformamide': 0.944 / 73.09 * 1000,
 'ethanol': 0.789 / 46.07 * 1000,
 'THF': 0.889 / 72.11 * 1000,
 'NH3(aq)': np.nan}

df = pd.read_excel('useful_data.xlsx')

df = (
    df.loc[df['morphology'].notna()]
    .loc[lambda d: ~d['morphology'].str.contains('film')]
    .loc[lambda d: ~d['morphology'].str.contains('membrane')]
    .loc[lambda d: ~d['morphology'].str.contains('hollow')]
    .loc[lambda d: ~d['morphology'].str.contains('plate')]
    .assign(temperature_unit = lambda d: d['temperature_unit'].map(temp_unit_mapping))
    .assign(reaction_time_unit = lambda d: d['reaction_time_unit'].map(time_unit_mapping))
    .assign(temperature_C = lambda d: np.where(d['temperature_unit'] == 'K',d['temperature_value']-273.15,d['temperature_value']))
    .assign(time_min = lambda d: np.where(d['reaction_time_unit'] == 'd',d['reaction_time_value']*24*60,np.nan))
    .assign(time_min = lambda d: np.where(d['reaction_time_unit'] == 'h',d['reaction_time_value']*60,d['time_min']))
    .assign(time_min = lambda d: np.where(d['reaction_time_unit'] == 'm',d['reaction_time_value'],d['time_min']))
    .assign(time_min = lambda d: np.where(d['reaction_time_unit'] == 's',d['reaction_time_value']/60,d['time_min']))
    .assign(size_nm = lambda d: np.where(d['size_unit'] == 'μm',d['size_value']*1000,np.nan))
    .assign(size_nm = lambda d: np.where(d['size_unit'] == 'µm',d['size_value']*1000,d['size_nm']))
    .assign(size_nm = lambda d: np.where(d['size_unit'] == 'nm',d['size_value'],d['size_nm']))
    .assign(zn_source = lambda d: np.where(d['metal_source_name'].notna(),d['metal_source_name'].map(zn_source),'Unknown'))
    .assign(zn_source_mass = lambda d: np.where(~d['zn_source'].str.contains('Unknown'),d['zn_source'].map(zn_source_mass_mapping),np.nan))
    .assign(solvent_name = lambda d: np.where(d['solvent_name'].notna(),d['solvent_name'].map(solvent),'Unknown'))
    .assign(solvent_unit = lambda d: np.where(d['solvent_volume_unit'].notna(),d['solvent_volume_unit'].map(solvent_volume_unit),'Unknown'))
    .assign(solvent = lambda d: np.where(d['solvent_unit'] == 'ml', d['solvent_volume_value'],np.nan))
    .assign(solvent = lambda d: np.where(d['solvent_unit'] == 'μl', d['solvent_volume_value']/1000,d['solvent']))
    .assign(zn_source_unit = lambda d: np.where(d['metal_source_amount_unit'].notna(),d['metal_source_amount_unit'].map(amount_unit_mapping),'Unknown'))
    .assign(zn_source_mmol = lambda d: np.where(d['zn_source_unit'] == 'mM',d['metal_source_amount_value'],np.nan))
    .assign(zn_source_mmol = lambda d: np.where(d['zn_source_unit'] == 'mmol',d['metal_source_amount_value'],d['zn_source_mmol'] ))
    .assign(zn_source_mmol = lambda d: np.where(d['zn_source_unit'] == 'ml',(d['metal_source_amount_value']*d['metal_source_concentration_value'])/d['zn_source_mass']*1000 ,d['zn_source_mmol'] ))
    .assign(zn_source_mmol = lambda d: np.where(d['zn_source_unit'] == 'mg',d['metal_source_amount_value']/d['zn_source_mass'],d['zn_source_mmol'] ))
    .assign(zn_source_mmol = lambda d: np.where(d['zn_source_unit'] == 'g',(d['metal_source_amount_value']/d['zn_source_mass'])*1000,d['zn_source_mmol'] ))
    .assign(hmim_unit = lambda d: np.where(d['ligand_amount_unit'].notna(),d['ligand_amount_unit'].map(amount_unit_mapping),'Unknown'))
    .assign(ligand_amount_value = lambda d: pd.to_numeric(d['ligand_amount_value'],errors='coerce'))
    .assign(hmim_mmol = lambda d: np.where(d['hmim_unit'] == 'mM',d['ligand_amount_value'],np.nan))
    .assign(hmim_mmol = lambda d: np.where(d['hmim_unit'] == 'mmol',d['ligand_amount_value'],d['hmim_mmol']))
    .assign(hmim_mmol = lambda d: np.where(d['hmim_unit'] == 'mg',d['ligand_amount_value']/HMIM_MASS,d['hmim_mmol']))
    .assign(hmim_mmol = lambda d: np.where(d['hmim_unit'] == 'g',(d['ligand_amount_value']/HMIM_MASS)*1000,d['hmim_mmol']))
    .assign(
        counterion = lambda d: d['zn_source'].map(counterion_mapping),
        stirring = lambda d: d['stirring_method'].map(stirring),
        additive = lambda d: d['additive_or_template_name'].map(additive_name),
        ratio = lambda d: d['zn_source_mmol']/d['hmim_mmol'],
        solvent = lambda d: d['solvent'] * d['solvent_name'].map(ml_to_mmol)
    )
    .loc[:,['DOI','solvent_name','solvent','zn_source','zn_source_mmol','hmim_mmol','additive','stirring','temperature_C','time_min','size_nm','counterion','ratio']]
    .pipe( lambda d: d.dropna(subset=[c for c in d.columns if c != 'additive'],how='any',ignore_index=True))
    .fillna({'additive':'not given'})

)   
df.to_csv('clean.csv',index=False)
df

,DOI,solvent_name,solvent,zn_source,zn_source_mmol,hmim_mmol,additive,stirring,temperature_C,time_min,size_nm,counterion,ratio
0,10.1021/acsami.9b23314,methanol,499.37578,Zn(NO3)2·6H2O,0.537869,4.993910,not given,Yes,50.0,180,48.0,NO3-,0.107705
1,10.1039/d4nr01102c,water,888.888889,Zn(NO3)2·6H2O,0.134467,11.059683,Cetyltrimethylammonium bromide,Yes,25.0,120,150.0,NO3-,0.012158
2,10.1016/j.ijbiomac.2024.135465,water,5555.555556,Zn(NO3)2,7.412883,39.585871,not given,Yes,25.0,40,354.8,NO3-,0.187261
3,10.1021/acsomega.3c07606,methanol,1248.439451,Zn(NO3)2·6H2O,1.017918,4.003654,not given,Yes,25.0,1440,150.0,NO3-,0.254247
4,10.1039/c7nr07949d,water,1111.111111,Zn(NO3)2·6H2O,0.050425,0.304507,not given,No,0.0,0.333333,45.0,NO3-,0.165597
...,...,...,...,...,...,...,...,...,...,...,...,...,...
243,10.1016/j.micromeso.2025.113513,THF,51.779226,Zn(NO3)2·6H2O,0.080008,0.239951,PS153-b-PEO114,Yes,25.0,720,200.0,NO3-,0.333435
244,10.1016/j.micromeso.2025.113513,THF,51.779226,Zn(NO3)2·6H2O,0.080008,0.239951,PS153-b-PEO114,Yes,25.0,720,500.0,NO3-,0.333435
245,10.1016/j.memsci.2025.125060,water,1666.666667,Zn(CH3COO)2·2H2O,0.419115,24.969549,not given,Yes,20.0,360,1000.0,CH3COO-,0.016785
246,10.1016/j.memsci.2025.125060,methanol,749.06367,Zn(CH3COO)2·2H2O,0.419115,24.969549,not given,Yes,20.0,360,150.0,CH3COO-,0.016785


In [3]:
df = pd.read_csv('clean.csv')

df_ = (
    df
    #.loc[lambda d: d['solvent_name'] == 'water']
    .rename({'ration':'ratio'},axis=1)
    #.loc[lambda d: d['size_nm']<1000]
    .assign(zn_source_mM = lambda d: d['zn_source_mmol']/d['solvent'],
            hmim_mM = lambda d: d['hmim_mmol']/d['solvent']
            )
    #.loc[lambda d: d['counterion'].map(d['counterion'].value_counts().to_dict()) > 1 ]
    .loc[:,['DOI', 'solvent_name', 'solvent', 'zn_source', 'zn_source_mmol',
       'hmim_mmol', 'stirring', 'temperature_C', 'time_min', 'size_nm',
       'additive', 'ratio', 'counterion', 'zn_source_mM', 'hmim_mM']]
    .assign(old = False)
    )
df_

,DOI,solvent_name,solvent,zn_source,zn_source_mmol,hmim_mmol,stirring,temperature_C,time_min,size_nm,additive,ratio,counterion,zn_source_mM,hmim_mM,old
0,10.1021/acsami.9b23314,methanol,499.375780,Zn(NO3)2·6H2O,0.537869,4.993910,Yes,50.0,180.000000,48.0,not given,0.107705,NO3-,0.001077,0.010000,False
1,10.1039/d4nr01102c,water,888.888889,Zn(NO3)2·6H2O,0.134467,11.059683,Yes,25.0,120.000000,150.0,Cetyltrimethylammonium bromide,0.012158,NO3-,0.000151,0.012442,False
2,10.1016/j.ijbiomac.2024.135465,water,5555.555556,Zn(NO3)2,7.412883,39.585871,Yes,25.0,40.000000,354.8,not given,0.187261,NO3-,0.001334,0.007125,False
3,10.1021/acsomega.3c07606,methanol,1248.439451,Zn(NO3)2·6H2O,1.017918,4.003654,Yes,25.0,1440.000000,150.0,not given,0.254247,NO3-,0.000815,0.003207,False
4,10.1039/c7nr07949d,water,1111.111111,Zn(NO3)2·6H2O,0.050425,0.304507,No,0.0,0.333333,45.0,not given,0.165597,NO3-,0.000045,0.000274,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
243,10.1016/j.micromeso.2025.113513,THF,51.779226,Zn(NO3)2·6H2O,0.080008,0.239951,Yes,25.0,720.000000,200.0,PS153-b-PEO114,0.333435,NO3-,0.001545,0.004634,False
244,10.1016/j.micromeso.2025.113513,THF,51.779226,Zn(NO3)2·6H2O,0.080008,0.239951,Yes,25.0,720.000000,500.0,PS153-b-PEO114,0.333435,NO3-,0.001545,0.004634,False
245,10.1016/j.memsci.2025.125060,water,1666.666667,Zn(CH3COO)2·2H2O,0.419115,24.969549,Yes,20.0,360.000000,1000.0,not given,0.016785,CH3COO-,0.000251,0.014982,False
246,10.1016/j.memsci.2025.125060,methanol,749.063670,Zn(CH3COO)2·2H2O,0.419115,24.969549,Yes,20.0,360.000000,150.0,not given,0.016785,CH3COO-,0.000560,0.033334,False


In [4]:
with open('mapping_dict.json','r') as f:
    mapping = json.load(f)

counterion_mapping = mapping.get('counterion',{})

df = pd.read_csv('8153451/ZIF8_Database.csv',sep='\t')

tmp = (
    df.assign(size_nm = lambda d: d['TEM_SEM_Diameter'].str.strip('*'))
    .assign(size_nm = lambda d: np.where(d['size_nm']=='-',d['SAXS_Diameter'].str.strip('*'),d['size_nm']))
    .assign(size_nm = lambda d: np.where(d['size_nm']=='-',d['Scherrer_equation_Diameter'].str.strip('*'),d['size_nm']))
    .assign(size_nm = lambda d: np.where(d['size_nm']=='-',d['DLS_Diameter'],d['size_nm']))
    .assign(size_nm = lambda d: np.where(d['size_nm']=='-',np.nan,d['size_nm']))
    .assign(size_nm = lambda d: pd.to_numeric(d['size_nm'],errors='coerce'))
    .assign(
            zn_source = lambda d: d['Zinc_salt'],
            zn_source_mM = lambda d: d['Zinc'],
            hmim_mM=lambda d: d['HmIm'],
            temperature_C=lambda d:d['Temperature'].replace('RT',25),
            time_min = lambda d: d['Reaction_time'],
            solvent_name =lambda d: d['Solvent_name'],
            solvent = lambda d: d['Solvent'],
            stirring = lambda d: d['Stirring']
            )
    .loc[:,['DOI','solvent_name','solvent','zn_source','zn_source_mM','hmim_mM','stirring','temperature_C','time_min','size_nm','Modulador']]
    .dropna(subset=['size_nm'])
    .assign(ratio = lambda d: d['zn_source_mM']/d['hmim_mM'],
            counterion = lambda d: d['zn_source'].map(counterion_mapping)
            )
    #.loc[lambda d: d['solvent_name'] == 'Methanol',:]
    .rename({'zn_source_mM':'zn_source_mmol','hmim_mM':'hmim_mmol','Modulador':'additive'},axis=1)
    #.loc[lambda d: d['size_nm'] < 1000]
    .assign(zn_source_mM = lambda d: d['zn_source_mmol']/d['solvent'],
            hmim_mM = lambda d: d['hmim_mmol']/d['solvent']
            )
    #.loc[lambda d: d['counterion'].map(d['counterion'].value_counts().to_dict()) > 1 ]
    .loc[:,['DOI', 'solvent_name', 'solvent', 'zn_source', 'zn_source_mmol',
       'hmim_mmol', 'stirring', 'temperature_C', 'time_min', 'size_nm',
       'additive', 'ratio', 'counterion', 'zn_source_mM', 'hmim_mM']]
    .assign(old = True)
)
tmp

,DOI,solvent_name,solvent,zn_source,zn_source_mmol,hmim_mmol,stirring,temperature_C,time_min,size_nm,additive,ratio,counterion,zn_source_mM,hmim_mM,old
0,10.1021/cm900166h,Methanol,9888.000,Zn(NO3)2·6H2O,9.870,79.040,Yes,25,60,40.0,-,0.124873,NO3-,0.000998,0.007994,True
1,10.1021/cm900166h,DMF,2325.000,Zn(NO3)2·6H2O,2.000,16.000,-,25,1080,40.0,-,0.125000,NO3-,0.000860,0.006882,True
2,10.1021/cm103571y,Methanol,2472.000,Zn(NO3)2·6H2O,2.469,9.874,Initial,25,1440,65.0,-,0.250051,NO3-,0.000999,0.003994,True
3,10.1021/cm103571y,Methanol,2472.000,Zn(NO3)2·6H2O,2.469,9.874,Initial,25,1440,3200.0,9.874,0.250051,NO3-,0.000999,0.003994,True
4,10.1021/cm103571y,Methanol,2472.000,Zn(NO3)2·6H2O,2.469,9.874,Initial,25,1440,1140.0,9.874,0.250051,NO3-,0.000999,0.003994,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
249,10.1016/j.micromeso.2020.110761,Methanol,1235.955,Zn(NO3)2·6H2O,2.464,19.817,Yes,25,1440,45.0,-,0.124338,NO3-,0.001994,0.016034,True
250,10.1016/j.micromeso.2020.110761,Methanol,1235.955,Zn(NO3)2·6H2O,2.497,9.957,Yes,25,1440,120.0,1.313,0.250778,NO3-,0.002020,0.008056,True
251,10.1016/j.micromeso.2020.110761,Methanol,1235.955,Zn(NO3)2·6H2O,2.497,9.957,Yes,25,1440,250.0,2.179,0.250778,NO3-,0.002020,0.008056,True
252,10.1016/j.micromeso.2020.110761,Methanol,1235.955,Zn(NO3)2·6H2O,2.497,9.957,Yes,25,1440,450.0,2.548,0.250778,NO3-,0.002020,0.008056,True


In [ ]:
df = pd.concat([df_,tmp],axis=0,ignore_index=True)

Q1 = df['size_nm'].quantile(0.25)
Q3 = df['size_nm'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1 * IQR
upper_bound = Q3 + 1 * IQR

df = (
    df.assign(check = lambda d: d['DOI'].map((d.groupby('DOI')['old'].nunique()>1).to_dict()),
              temperature_C = lambda d: pd.to_numeric(d['temperature_C']),
              stirring = lambda d: d['stirring'].replace('-','No'),
              solvent_name = lambda d: d['solvent_name'].replace({'water':'Water','methanol':'Methanol'})
              )
    .loc[lambda d: ~(d['check'] & ~d['old'])]
    .loc[lambda d: ~(d['size_nm'] > upper_bound) | (d['size_nm'] < lower_bound)]
    .loc[lambda d: d['stirring'].map(d['stirring'].value_counts().to_dict()) > 1]
    .loc[lambda d: d['solvent_name'].map(d['solvent_name'].value_counts().to_dict()) > 1]
    .loc[lambda d: d['size_nm'] > 0]
    .reset_index(drop=True)
)
# df.to_csv('data.csv',index=False)
df

,DOI,solvent_name,solvent,zn_source,zn_source_mmol,hmim_mmol,stirring,temperature_C,time_min,size_nm,additive,ratio,counterion,zn_source_mM,hmim_mM,old,check
0,10.1021/acsami.9b23314,Methanol,499.375780,Zn(NO3)2·6H2O,0.537869,4.993910,Yes,50.0,180.000000,48.0,not given,0.107705,NO3-,0.001077,0.010000,False,False
1,10.1039/d4nr01102c,Water,888.888889,Zn(NO3)2·6H2O,0.134467,11.059683,Yes,25.0,120.000000,150.0,Cetyltrimethylammonium bromide,0.012158,NO3-,0.000151,0.012442,False,False
2,10.1016/j.ijbiomac.2024.135465,Water,5555.555556,Zn(NO3)2,7.412883,39.585871,Yes,25.0,40.000000,354.8,not given,0.187261,NO3-,0.001334,0.007125,False,False
3,10.1021/acsomega.3c07606,Methanol,1248.439451,Zn(NO3)2·6H2O,1.017918,4.003654,Yes,25.0,1440.000000,150.0,not given,0.254247,NO3-,0.000815,0.003207,False,False
4,10.1039/c7nr07949d,Water,1111.111111,Zn(NO3)2·6H2O,0.050425,0.304507,No,0.0,0.333333,45.0,not given,0.165597,NO3-,0.000045,0.000274,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
386,10.1016/j.micromeso.2020.110761,Methanol,1235.955000,Zn(NO3)2·6H2O,2.464000,19.817000,Yes,25.0,1440.000000,45.0,-,0.124338,NO3-,0.001994,0.016034,True,False
387,10.1016/j.micromeso.2020.110761,Methanol,1235.955000,Zn(NO3)2·6H2O,2.497000,9.957000,Yes,25.0,1440.000000,120.0,1.313,0.250778,NO3-,0.002020,0.008056,True,False
388,10.1016/j.micromeso.2020.110761,Methanol,1235.955000,Zn(NO3)2·6H2O,2.497000,9.957000,Yes,25.0,1440.000000,250.0,2.179,0.250778,NO3-,0.002020,0.008056,True,False
389,10.1016/j.micromeso.2020.110761,Methanol,1235.955000,Zn(NO3)2·6H2O,2.497000,9.957000,Yes,25.0,1440.000000,450.0,2.548,0.250778,NO3-,0.002020,0.008056,True,False


## Verify for Data
Remove data that should not appear through manual verification

In [1]:
import pandas as pd

In [3]:
df = pd.read_csv('Data/verify_data.csv')

df.loc[df['Verify']].drop(columns=['Verify']).to_csv('Data/fit_data.csv')